# TV6 — Thực nghiệm & Đánh giá định lượng

**Mục tiêu:** đánh giá kết quả phân loại mức độ ùn tắc giao thông và sai số vận tốc của hệ thống.

Notebook này thực hiện:
- Tính Accuracy, Precision, Recall và F1-score.
- Phân tích kết quả theo 4 mức độ ùn tắc.
- Vẽ Confusion Matrix.
- Tính sai số vận tốc MAE.
- So sánh F1-score với mục tiêu **85%**.
- Tổng hợp kết quả và đưa ra nhận xét.

> **Lưu ý:** Dữ liệu trong phần DEMO chỉ dùng để kiểm tra notebook. Không được xem là kết quả thực nghiệm chính thức của nhóm.


## 1. Import thư viện

Sử dụng các thư viện đã có trong `requirements.txt` của project.


In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
)

# Thêm thư mục gốc của project vào Python Path
THU_MUC_GOC = Path.cwd()

if str(THU_MUC_GOC) not in sys.path:
    sys.path.insert(0, str(THU_MUC_GOC))

from evaluation.metrics import (
    compute_classification_metrics,
    plot_confusion_matrix,
    compute_speed_mae,
)

print("Đã import thư viện thành công.")
print("Thư mục project:", THU_MUC_GOC)


## 2. Khai báo 4 mức độ ùn tắc

Tên lớp được giữ thống nhất với hệ thống TCI của project.


In [ ]:
cac_muc_do = [
    "1 - Thong thoang",
    "2 - Binh thuong",
    "3 - Un u",
    "4 - Tac nghen",
]

print("Các mức độ ùn tắc:")
for i, muc_do in enumerate(cac_muc_do, start=1):
    print(f"{i}. {muc_do}")


## 3. Chuẩn bị dữ liệu DEMO

Dữ liệu dưới đây chỉ dùng để kiểm tra toàn bộ quy trình đánh giá khi nhóm chưa có bộ Ground Truth/Prediction chính thức.

- `nhan_thuc_te`: Ground Truth — nhãn thực tế.
- `nhan_du_doan`: Prediction — nhãn do hệ thống dự đoán.


In [ ]:
# Dữ liệu DEMO để kiểm tra notebook
nhan_thuc_te = [
    cac_muc_do[0],
    cac_muc_do[1],
    cac_muc_do[2],
    cac_muc_do[3],
    cac_muc_do[3],
    cac_muc_do[2],
    cac_muc_do[0],
    cac_muc_do[1],
    cac_muc_do[2],
    cac_muc_do[3],
    cac_muc_do[1],
    cac_muc_do[3],
]

nhan_du_doan = [
    cac_muc_do[0],
    cac_muc_do[1],
    cac_muc_do[1],
    cac_muc_do[3],
    cac_muc_do[2],
    cac_muc_do[2],
    cac_muc_do[0],
    cac_muc_do[2],
    cac_muc_do[2],
    cac_muc_do[3],
    cac_muc_do[1],
    cac_muc_do[2],
]

print("Số mẫu Ground Truth:", len(nhan_thuc_te))
print("Số mẫu Prediction  :", len(nhan_du_doan))


## 4. Kiểm tra dữ liệu đánh giá

Kiểm tra số lượng mẫu và thống kê phân bố của từng mức độ ùn tắc.


In [ ]:
du_lieu_danh_gia = pd.DataFrame({
    "Nhan thuc te": nhan_thuc_te,
    "Nhan du doan": nhan_du_doan,
})

print("5 dòng đầu tiên:")
display(du_lieu_danh_gia.head())

print("\nSố lượng mẫu theo Ground Truth:")
print(
    du_lieu_danh_gia["Nhan thuc te"]
    .value_counts()
    .reindex(cac_muc_do, fill_value=0)
)

print("\nSố lượng mẫu theo Prediction:")
print(
    du_lieu_danh_gia["Nhan du doan"]
    .value_counts()
    .reindex(cac_muc_do, fill_value=0)
)


## 5. Trực quan hóa phân bố dữ liệu

Biểu đồ giúp kiểm tra nhanh dữ liệu có bị mất một lớp hoặc mất cân bằng quá lớn hay không.


In [ ]:
tan_suat_thuc_te = (
    du_lieu_danh_gia["Nhan thuc te"]
    .value_counts()
    .reindex(cac_muc_do, fill_value=0)
)

tan_suat_du_doan = (
    du_lieu_danh_gia["Nhan du doan"]
    .value_counts()
    .reindex(cac_muc_do, fill_value=0)
)

vi_tri = np.arange(len(cac_muc_do))
chieu_rong = 0.35

plt.figure(figsize=(10, 5))
plt.bar(
    vi_tri - chieu_rong / 2,
    tan_suat_thuc_te.values,
    chieu_rong,
    label="Ground Truth",
)
plt.bar(
    vi_tri + chieu_rong / 2,
    tan_suat_du_doan.values,
    chieu_rong,
    label="Prediction",
)

plt.xticks(vi_tri, cac_muc_do, rotation=20)
plt.ylabel("Số lượng mẫu")
plt.xlabel("Mức độ ùn tắc")
plt.title("Phân bố Ground Truth và Prediction")
plt.legend()
plt.tight_layout()
plt.show()


## 6. Tính các chỉ số Accuracy, Precision, Recall và F1-score

TV6 sử dụng các metric phân loại để đánh giá mức độ chính xác của hệ thống.


In [ ]:
ket_qua_metric = compute_classification_metrics(
    nhan_thuc_te,
    nhan_du_doan,
)

print("KẾT QUẢ ĐÁNH GIÁ")
print("-" * 45)
print(f"Accuracy          : {ket_qua_metric['accuracy']:.4f}")
print(f"Precision (Macro) : {ket_qua_metric['precision_macro']:.4f}")
print(f"Recall (Macro)    : {ket_qua_metric['recall_macro']:.4f}")
print(f"F1-score (Macro)  : {ket_qua_metric['f1_macro']:.4f}")
print(f"Precision Weighted : {ket_qua_metric['precision_weighted']:.4f}")
print(f"Recall Weighted    : {ket_qua_metric['recall_weighted']:.4f}")
print(f"F1-score Weighted  : {ket_qua_metric['f1_weighted']:.4f}")


## 7. Tạo bảng tổng hợp metric

Bảng này thuận tiện để đưa vào báo cáo thực nghiệm.


In [ ]:
bang_metric = pd.DataFrame({
    "Chi so": [
        "Accuracy",
        "Precision (Macro)",
        "Recall (Macro)",
        "F1-score (Macro)",
        "Precision (Weighted)",
        "Recall (Weighted)",
        "F1-score (Weighted)",
    ],
    "Gia tri": [
        ket_qua_metric["accuracy"],
        ket_qua_metric["precision_macro"],
        ket_qua_metric["recall_macro"],
        ket_qua_metric["f1_macro"],
        ket_qua_metric["precision_weighted"],
        ket_qua_metric["recall_weighted"],
        ket_qua_metric["f1_weighted"],
    ],
})

bang_metric["Phan tram"] = bang_metric["Gia tri"] * 100

display(
    bang_metric.style.format({
        "Gia tri": "{:.4f}",
        "Phan tram": "{:.2f}%",
    })
)


## 8. Classification Report

Phân tích Precision, Recall và F1-score riêng cho từng mức độ ùn tắc.


In [ ]:
bao_cao_phan_loai = classification_report(
    nhan_thuc_te,
    nhan_du_doan,
    labels=cac_muc_do,
    target_names=cac_muc_do,
    zero_division=0,
)

print(bao_cao_phan_loai)


## 9. Phân tích metric theo từng mức độ

Tạo bảng dữ liệu từ Classification Report để dễ quan sát lớp nào được dự đoán tốt và lớp nào còn nhầm lẫn.


In [ ]:
bao_cao_dict = classification_report(
    nhan_thuc_te,
    nhan_du_doan,
    labels=cac_muc_do,
    target_names=cac_muc_do,
    output_dict=True,
    zero_division=0,
)

bang_theo_lop = pd.DataFrame(
    bao_cao_dict
).T.loc[cac_muc_do, ["precision", "recall", "f1-score", "support"]]

bang_theo_lop.columns = [
    "Precision",
    "Recall",
    "F1-score",
    "So mau",
]

display(
    bang_theo_lop.style.format({
        "Precision": "{:.4f}",
        "Recall": "{:.4f}",
        "F1-score": "{:.4f}",
        "So mau": "{:.0f}",
    })
)


## 10. Confusion Matrix

- Hàng: Ground Truth.
- Cột: Prediction.
- Đường chéo chính: số mẫu được phân loại đúng.
- Ngoài đường chéo: các trường hợp hệ thống dự đoán nhầm.


In [ ]:
ma_tran = confusion_matrix(
    nhan_thuc_te,
    nhan_du_doan,
    labels=cac_muc_do,
)

bang_ma_tran = pd.DataFrame(
    ma_tran,
    index=cac_muc_do,
    columns=cac_muc_do,
)

print("Ma trận nhầm lẫn:")
display(bang_ma_tran)


## 11. Vẽ Confusion Matrix

Sử dụng hàm `plot_confusion_matrix()` trong `evaluation/metrics.py` để tạo và lưu hình kết quả.


In [ ]:
duong_dan_ma_tran = Path(
    "docs/figures/confusion_matrix.png"
)

duong_dan_da_luu = plot_confusion_matrix(
    nhan_thuc_te,
    nhan_du_doan,
    cac_muc_do,
    duong_dan_ma_tran,
)

print("Đã lưu Confusion Matrix tại:")
print(duong_dan_da_luu)


## 12. Hiển thị Confusion Matrix trực tiếp trong notebook


In [ ]:
plt.figure(figsize=(8, 6))
plt.imshow(ma_tran, interpolation="nearest")

plt.xticks(
    np.arange(len(cac_muc_do)),
    cac_muc_do,
    rotation=20,
)
plt.yticks(
    np.arange(len(cac_muc_do)),
    cac_muc_do,
)

plt.xlabel("Prediction")
plt.ylabel("Ground Truth")
plt.title("Ma trận nhầm lẫn mức độ ùn tắc")

nguong = ma_tran.max() / 2 if ma_tran.size else 0

for hang in range(ma_tran.shape[0]):
    for cot in range(ma_tran.shape[1]):
        plt.text(
            cot,
            hang,
            str(ma_tran[hang, cot]),
            ha="center",
            va="center",
            color="white" if ma_tran[hang, cot] > nguong else "black",
        )

plt.colorbar()
plt.tight_layout()
plt.show()


## 13. Chuẩn bị dữ liệu vận tốc

TV4 cung cấp vận tốc ước lượng từ Optical Flow. TV6 đối sánh vận tốc dự đoán với vận tốc thực tế/chuẩn để tính MAE.

Trong phần DEMO này, dữ liệu chỉ nhằm kiểm tra công thức.


In [ ]:
van_toc_thuc_te = np.array([
    20,
    30,
    40,
    35,
    25,
    45,
    28,
    32,
])

van_toc_du_doan = np.array([
    22,
    27,
    43,
    32,
    26,
    42,
    30,
    29,
])

print("Số mẫu vận tốc:", len(van_toc_thuc_te))
print("Vận tốc thực tế :", van_toc_thuc_te)
print("Vận tốc dự đoán :", van_toc_du_doan)


## 14. Tính sai số tuyệt đối và MAE

Công thức:

**MAE = trung bình của |vận tốc thực tế − vận tốc dự đoán|**

MAE càng nhỏ thì sai số ước lượng vận tốc càng thấp.


In [ ]:
sai_so_tuyet_doi = np.abs(
    van_toc_thuc_te - van_toc_du_doan
)

mae = compute_speed_mae(
    van_toc_thuc_te,
    van_toc_du_doan,
)

bang_van_toc = pd.DataFrame({
    "Mau": np.arange(1, len(van_toc_thuc_te) + 1),
    "Van toc thuc te": van_toc_thuc_te,
    "Van toc du doan": van_toc_du_doan,
    "Sai so tuyet doi": sai_so_tuyet_doi,
})

display(
    bang_van_toc.style.format({
        "Van toc thuc te": "{:.2f}",
        "Van toc du doan": "{:.2f}",
        "Sai so tuyet doi": "{:.2f}",
    })
)

print(f"MAE vận tốc: {mae:.4f}")


## 15. Biểu đồ Ground Truth và Prediction của vận tốc


In [ ]:
plt.figure(figsize=(10, 5))

plt.plot(
    range(1, len(van_toc_thuc_te) + 1),
    van_toc_thuc_te,
    marker="o",
    label="Ground Truth",
)

plt.plot(
    range(1, len(van_toc_du_doan) + 1),
    van_toc_du_doan,
    marker="x",
    label="Prediction",
)

plt.xlabel("Mẫu dữ liệu")
plt.ylabel("Vận tốc")
plt.title("So sánh vận tốc thực tế và vận tốc dự đoán")
plt.legend()
plt.grid()
plt.tight_layout()
plt.show()


## 16. Phân tích sai số vận tốc

Quan sát sai số của từng mẫu để tìm các trường hợp hệ thống ước lượng vận tốc lệch nhiều.


In [ ]:
bang_van_toc["Ty le sai so (%)"] = (
    bang_van_toc["Sai so tuyet doi"]
    / bang_van_toc["Van toc thuc te"]
    * 100
)

print("Mẫu có sai số lớn nhất:")
mau_sai_so_lon_nhat = bang_van_toc.loc[
    bang_van_toc["Sai so tuyet doi"].idxmax()
]

display(
    mau_sai_so_lon_nhat.to_frame().T.style.format({
        "Van toc thuc te": "{:.2f}",
        "Van toc du doan": "{:.2f}",
        "Sai so tuyet doi": "{:.2f}",
        "Ty le sai so (%)": "{:.2f}",
    })
)

print(f"MAE tổng thể: {mae:.4f}")


## 17. Kiểm tra mục tiêu F1-score ≥ 85%

Theo mục tiêu của project, F1-score được dùng để đánh giá chất lượng phân loại mức độ ùn tắc.

Phần này chỉ là phép kiểm tra mục tiêu. Với dữ liệu DEMO, kết quả không đại diện cho kết quả chính thức của nhóm.


In [ ]:
MUC_TIEU_F1 = 0.85

f1_thuc_te = ket_qua_metric["f1_macro"]

print(f"F1-score Macro : {f1_thuc_te:.2%}")
print(f"Mục tiêu        : {MUC_TIEU_F1:.2%}")

if f1_thuc_te >= MUC_TIEU_F1:
    print("KẾT LUẬN DEMO: ĐẠT mục tiêu F1 >= 85%.")
else:
    print("KẾT LUẬN DEMO: CHƯA ĐẠT mục tiêu F1 >= 85%.")

print("\nLưu ý: Đây là dữ liệu DEMO, không phải kết quả thực nghiệm chính thức.")


## 18. Tổng hợp kết quả thực nghiệm

Tổng hợp các chỉ số quan trọng nhất của TV6 thành một bảng.


In [ ]:
tong_hop = pd.DataFrame({
    "Chi tieu": [
        "Accuracy",
        "Precision Macro",
        "Recall Macro",
        "F1-score Macro",
        "F1-score Weighted",
        "Speed MAE",
    ],
    "Ket qua": [
        ket_qua_metric["accuracy"],
        ket_qua_metric["precision_macro"],
        ket_qua_metric["recall_macro"],
        ket_qua_metric["f1_macro"],
        ket_qua_metric["f1_weighted"],
        mae,
    ],
    "Don vi": [
        "%",
        "%",
        "%",
        "%",
        "%",
        "don vi van toc",
    ],
})

display(tong_hop.style.format({
    "Ket qua": "{:.4f}",
}))


## 19. Kết luận

### Đối với phân loại ùn tắc
- Accuracy cho biết tỷ lệ dự đoán đúng trên toàn bộ mẫu.
- Precision cho biết mức độ chính xác của các dự đoán.
- Recall cho biết khả năng phát hiện đúng các mẫu của từng lớp.
- F1-score cân bằng giữa Precision và Recall.
- Confusion Matrix cho thấy các mức độ ùn tắc thường bị nhầm lẫn với nhau.

### Đối với vận tốc
- MAE cho biết sai số tuyệt đối trung bình giữa vận tốc thực tế và vận tốc dự đoán.
- MAE càng nhỏ thì kết quả ước lượng vận tốc càng gần với dữ liệu chuẩn.

> **Kết luận cuối cùng của nhóm phải được cập nhật lại khi có Ground Truth và Prediction thực tế.**
